# Clase 101 — Regresión y clasificación con MLP

Un MLP resuelve los **tres problemas tabulares** estándar. La clave es elegir bien
la **activación de salida** y la **loss** para cada caso:

| Problema | Salida | Loss |
|---|---|---|
| Regresión | `linear` | `mse` |
| Clasificación binaria | `sigmoid` | `binary_crossentropy` |
| Clasificación multiclase | `softmax` | `sparse_categorical_crossentropy` |

Requiere: `tensorflow` / `keras`, `scikit-learn` (se ejecuta en Colab con GPU).

## 1. Regresión: California Housing con `Normalization()`

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

np.random.seed(42); keras.utils.set_random_seed(42)

X, y = fetch_california_housing(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

norm = layers.Normalization()      # aprende media/desvío de los inputs
norm.adapt(X_tr)

reg = keras.Sequential([
    keras.Input(shape=(X.shape[1],)),
    norm,
    layers.Dense(64, activation="relu"),
    layers.Dense(32, activation="relu"),
    layers.Dense(1),               # salida linear para regresión
])
reg.compile(optimizer="adam", loss="mse", metrics=["mae"])
reg.fit(X_tr, y_tr, validation_split=0.2, epochs=10, batch_size=64, verbose=0)
mse, mae = reg.evaluate(X_te, y_te, verbose=0)
print(f"regresión -> MAE en test: {mae:.3f} (en unidades de $100k)")

## 2. Clasificación binaria: Breast Cancer (`sigmoid` + BCE)

In [ ]:
from sklearn.datasets import load_breast_cancer

Xb, yb = load_breast_cancer(return_X_y=True)
Xb_tr, Xb_te, yb_tr, yb_te = train_test_split(Xb, yb, test_size=0.2, random_state=42)

nb = layers.Normalization(); nb.adapt(Xb_tr)
clf_bin = keras.Sequential([
    keras.Input(shape=(Xb.shape[1],)),
    nb,
    layers.Dense(32, activation="relu"),
    layers.Dense(16, activation="relu"),
    layers.Dense(1, activation="sigmoid"),    # probabilidad de clase positiva
])
clf_bin.compile(optimizer="adam", loss="binary_crossentropy",
                metrics=["accuracy", keras.metrics.AUC(name="auc")])
clf_bin.fit(Xb_tr, yb_tr, validation_split=0.2, epochs=10, verbose=0)
loss, acc, auc = clf_bin.evaluate(Xb_te, yb_te, verbose=0)
print(f"binario -> accuracy: {acc:.3f} | AUC: {auc:.3f}")

## 3. Multiclase: Fashion-MNIST (`softmax` + sparse CE)

In [ ]:
(Xf_tr, yf_tr), (Xf_te, yf_te) = keras.datasets.fashion_mnist.load_data()
Xf_tr = Xf_tr.astype("float32") / 255.0     # normalizar imágenes a [0, 1]
Xf_te = Xf_te.astype("float32") / 255.0

clf_multi = keras.Sequential([
    keras.Input(shape=(28, 28)),
    layers.Flatten(),                        # 28x28 -> 784
    layers.Dense(256, activation="relu"),
    layers.Dense(128, activation="relu"),
    layers.Dense(10, activation="softmax"),  # 10 clases (una por prenda)
])
clf_multi.compile(optimizer="adam",
                  loss="sparse_categorical_crossentropy",   # labels enteros
                  metrics=["accuracy"])
hist = clf_multi.fit(Xf_tr, yf_tr, validation_split=0.1,
                     epochs=10, batch_size=64, verbose=0)
test_loss, test_acc = clf_multi.evaluate(Xf_te, yf_te, verbose=0)
print(f"multiclase -> accuracy en test: {test_acc:.3f}")

## 4. Curvas de aprendizaje: detectar overfitting

In [ ]:
import matplotlib.pyplot as plt

h = hist.history
plt.figure(figsize=(6, 4))
plt.plot(h["loss"], label="train loss")
plt.plot(h["val_loss"], label="val loss")
plt.xlabel("época"); plt.ylabel("loss"); plt.legend()
plt.title("Cuando val_loss sube y train_loss baja -> overfitting")
plt.tight_layout(); plt.show()

## 5. `EarlyStopping`: cortar cuando val_loss deja de mejorar

In [ ]:
early = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=5, restore_best_weights=True)

modelo = keras.Sequential([
    keras.Input(shape=(28, 28)),
    layers.Flatten(),
    layers.Dense(300, activation="relu"),
    layers.Dense(100, activation="relu"),
    layers.Dense(10, activation="softmax"),
])
modelo.compile(optimizer="adam", loss="sparse_categorical_crossentropy",
               metrics=["accuracy"])
h2 = modelo.fit(Xf_tr, yf_tr, validation_split=0.1,
                epochs=100, batch_size=64, callbacks=[early], verbose=0)
print(f"EarlyStopping cortó en la época {len(h2.history['loss'])} (de 100 posibles)")
print("restore_best_weights=True deja el modelo en su mejor punto de val_loss.")

## Ejercicios

1. **MAE de regresión**: reportá el MAE de California Housing y convertilo a dólares
   (`mae * 100_000`). ¿Es un error razonable para el precio de una casa?
2. **AUC vs accuracy**: en Breast Cancer, explicá por qué el AUC puede ser más
   informativo que la accuracy cuando las clases están desbalanceadas.
3. **Matriz de confusión**: sobre Fashion-MNIST, calculá
   `sklearn.metrics.confusion_matrix` y verificá que `Shirt`/`T-shirt`/`Pullover`
   son las prendas más confundidas.
4. **Sin EarlyStopping**: entrená 50 épocas sin callback e identificá visualmente la
   época donde arranca el overfitting.

## Conclusiones

- El **tipo de problema** define salida + loss: linear/MSE, sigmoid/BCE, softmax/sparse-CE.
- **Normalizar** los inputs (o `/255` en imágenes) es casi siempre obligatorio.
- Las **curvas** `loss` vs `val_loss` diagnostican underfitting y overfitting de un vistazo.
- **EarlyStopping** con `restore_best_weights=True` es la red de seguridad estándar.
- El **test set** se toca una sola vez, al final, para reportar el resultado real.